# Multi-armed bandits — interactive companion

Companion to [Post 1a: Multi-armed bandits](../posts/01a-multi-armed-bandits.qmd).
The post derives the regret bounds and explains the exploration-exploitation
tradeoff; this notebook lets you run all three classical algorithms and watch
their behavior.

**What you'll do (≈ 15 minutes):**
1. Implement ε-greedy, UCB1, and Thompson sampling in ~30 lines.
2. Race them on stationary Bernoulli bandits — see who wins.
3. Switch to a non-stationary problem and watch Thompson collapse.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.bandits import BernoulliBandit
rng = np.random.default_rng(0)

## 1. The three classical algorithms

Each is about 10 lines. All three solve "which arm do I pull next?" given
the history of `(arm, reward)` pairs so far.

In [ ]:
class EpsilonGreedy:
    def __init__(self, n_arms, eps=0.1):
        self.n_arms, self.eps = n_arms, eps
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
    def select(self, rng):
        if rng.random() < self.eps:
            return int(rng.integers(self.n_arms))
        return int(self.values.argmax())
    def update(self, arm, reward):
        self.counts[arm] += 1
        self.values[arm] += (reward - self.values[arm]) / self.counts[arm]


class UCB1:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
        self.t = 0
    def select(self, rng):
        self.t += 1
        # Pull each arm once to start.
        if (self.counts == 0).any():
            return int(np.argmin(self.counts))
        bonus = np.sqrt(2 * np.log(self.t) / self.counts)
        return int((self.values + bonus).argmax())
    def update(self, arm, reward):
        self.counts[arm] += 1
        self.values[arm] += (reward - self.values[arm]) / self.counts[arm]


class ThompsonSampling:
    '''Beta-Bernoulli Thompson sampling: maintains posterior over each arm's mean.'''
    def __init__(self, n_arms):
        self.alpha = np.ones(n_arms)   # successes + 1
        self.beta = np.ones(n_arms)    # failures + 1
    def select(self, rng):
        samples = rng.beta(self.alpha, self.beta)
        return int(samples.argmax())
    def update(self, arm, reward):
        self.alpha[arm] += reward
        self.beta[arm] += 1 - reward


print("Three classical bandit algorithms, in 30 lines total.")

## 2. Stationary race

Bernoulli bandit with 5 arms, true means `[0.1, 0.2, 0.3, 0.5, 0.7]`.
Run each algorithm for 2000 steps and plot cumulative regret.

In [ ]:
true_means = np.array([0.1, 0.2, 0.3, 0.5, 0.7])
n_steps = 2000
n_seeds = 20

def run(agent_factory):
    regrets = np.zeros((n_seeds, n_steps))
    for seed in range(n_seeds):
        rng_local = np.random.default_rng(seed)
        agent = agent_factory()
        cum_regret = 0.0
        best_mean = true_means.max()
        for t in range(n_steps):
            arm = agent.select(rng_local)
            reward = float(rng_local.random() < true_means[arm])
            agent.update(arm, reward)
            cum_regret += best_mean - true_means[arm]
            regrets[seed, t] = cum_regret
    return regrets

results = {
    r"$\varepsilon$-greedy ($\varepsilon=0.1$)": run(lambda: EpsilonGreedy(5, eps=0.1)),
    "UCB1": run(lambda: UCB1(5)),
    "Thompson sampling": run(lambda: ThompsonSampling(5)),
}

for name, r in results.items():
    mean, std = r.mean(0), r.std(0)
    plt.plot(mean, label=name)
    plt.fill_between(np.arange(n_steps), mean - std, mean + std, alpha=0.18)
plt.xlabel("step"); plt.ylabel("cumulative regret")
plt.title("Stationary 5-arm Bernoulli bandit (20 seeds)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

On stationary problems you should see Thompson sampling and UCB1 with logarithmic
regret (the slope decays), while ε-greedy with fixed ε has linear regret (the slope
stays roughly constant) — it keeps wasting exploration on suboptimal arms forever.

### Try this
- Set `eps = 0.01`. Does ε-greedy now beat UCB1 and TS?
- Use `eps_t = 1 / sqrt(t)` (decaying ε). This is the classical "right" schedule. Modify the agent and re-run.
- Try `true_means = np.array([0.49, 0.51])` (very similar arms). What happens?

## 3. Non-stationarity: Thompson's confidence trap

Now the world changes at step 1000 — the best arm becomes the worst.
A confident Thompson sampler has accumulated thousands of successes for
that arm and now refuses to explore. UCB1's $\sqrt{\log t / n}$ bonus
grows again as time passes — it never fully stops exploring.

In [ ]:
def run_nonstationary(agent_factory):
    regrets = np.zeros((n_seeds, n_steps))
    for seed in range(n_seeds):
        rng_local = np.random.default_rng(seed)
        agent = agent_factory()
        # Two reward regimes.
        means_a = np.array([0.1, 0.2, 0.3, 0.5, 0.7])
        means_b = means_a[::-1].copy()  # reversed: now arm 0 is best
        cum_regret = 0.0
        for t in range(n_steps):
            cur_means = means_a if t < 1000 else means_b
            best = cur_means.max()
            arm = agent.select(rng_local)
            reward = float(rng_local.random() < cur_means[arm])
            agent.update(arm, reward)
            cum_regret += best - cur_means[arm]
            regrets[seed, t] = cum_regret
    return regrets

results = {
    r"$\varepsilon$-greedy ($\varepsilon=0.1$)": run_nonstationary(lambda: EpsilonGreedy(5, eps=0.1)),
    "UCB1": run_nonstationary(lambda: UCB1(5)),
    "Thompson sampling": run_nonstationary(lambda: ThompsonSampling(5)),
}

for name, r in results.items():
    mean, std = r.mean(0), r.std(0)
    plt.plot(mean, label=name)
    plt.fill_between(np.arange(n_steps), mean - std, mean + std, alpha=0.18)
plt.axvline(1000, color="black", linestyle="--", alpha=0.5)
plt.text(1010, plt.gca().get_ylim()[1]*0.9, "world flips", fontsize=10)
plt.xlabel("step"); plt.ylabel("cumulative regret")
plt.title("Non-stationary bandit — arms reverse at step 1000")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Honest result** (also in Post 1a §6): Thompson sampling — usually the best —
loses dramatically here. Its confidence in the old best arm is too strong.
UCB1's bonus term decays as $1/\sqrt{n}$ but never reaches zero, so it
continues to revisit other arms.

This is *the* practical reason RL practitioners often prefer UCB-style
exploration in non-stationary settings (online ad serving, recommendation
systems with changing user preferences). The Bayesian agent is overconfident.

### Try this
- Move the switch to step 500 (less prior data for TS). Does TS recover faster?
- Increase the gap between the new best and second-best arm. Does TS recover faster?
- Implement *sliding-window UCB* (only use the last 200 steps) and add it as a fourth agent.

## What's next

You've seen the classical bandit trio. Two natural extensions:

- **Contextual bandits** (Post 1b): the reward also depends on a context
  vector. Now you can do *targeted* exploration.
- **Full MDPs** (Post 1c): each action transitions you to a new state,
  and rewards may be delayed. Bellman's equation enters.

Open [`01b-contextual-bandits.ipynb`](01b-contextual-bandits.ipynb) for the next step.